### What is an Agent
- Agent = LLM + Tools + Decision Loop
- It can:
  - Decide what to do
  - Use tools
  - Take multiple steps

#### Modern LangChain Agents

- Ignore old initialize_agent for now
- create_openai_tools_agent
- AgentExecutor

- OLD (deprecated style)
  - initialize_agent(...)
  - Hidden logic
  - Hard to control
  - Not aligned with modern LLM APIs (like tool calling)
- NEW (Modern Standard)
  - create_openai_tools_agent
  - AgentExecutor
- LLM decides:
  - Should I answer directly?
  - OR call a tool?
  - Then AgentExecutor executes that decision




- create_openai_tools_agent

```python
from langchain.tools import tool
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# 1. Define tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

tools = [multiply]

# 2. LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Prompt (IMPORTANT)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # required for agent thinking
])

# 4. Create agent
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# 5. Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# 6. Run
response = agent_executor.invoke({
    "input": "What is 5 multiplied by 6?"
})

print(response)
```

____
____

##### **OLD: initialize_agent = opaque + fake tool calling**

```raw
LLM outputs TEXT like:
"Action: multiply\nAction Input: 5,6"

- Then LangChain:
   - parses that text (!!)
   - figures out which tool to call
   - executes it
   - This is string parsing based
   - fragile, hacky, error-prone


##### **NEW: create_openai_tools_agent = native tool calling**

- Modern LLMs support structured tool calling
- Now the LLM outputs:

```raw
{
  "tool_calls": [
    {
      "name": "multiply",
      "arguments": { "a": 5, "b": 6 }
    }
  ]
}

- No parsing
- No guessing
- Direct JSON from model

##### OLD world:
  - LLM → text → parse → tool
##### NEW world:
  - LLM → structured tool call → tool

- initialize_agent
  - everything bundled together
  - less control
  - “magic”

| Aspect                             | `initialize_agent` (OLD)              | `create_openai_tools_agent` (NEW)  |
| ---------------------------------- | ------------------------------------- | ---------------------------------- |
| **Core mechanism**                 | Text-based reasoning                  | Native tool calling (structured)   |
| **How tools are called**           | LLM writes text like `"Action: tool"` | LLM outputs JSON tool calls        |
| **Reliability**                    | ❌ Fragile (depends on text format)    | ✅ Robust (schema-based)            |
| **Parsing needed**                 | ✅ Yes (LangChain parses text)         | ❌ No parsing needed                |
| **Error handling**                 | ❌ Breaks easily on format deviation   | ✅ Stable and predictable           |
| **Alignment with modern LLM APIs** | ❌ Not aligned                         | ✅ Fully aligned                    |
| **Transparency**                   | ❌ Hidden “magic”                      | ✅ Explicit components              |
| **Control level**                  | ❌ Low                                 | ✅ High                             |
| **Separation of concerns**         | ❌ Mixed (agent + execution bundled)   | ✅ Agent + Executor separated       |
| **Extensibility**                  | ❌ Hard to extend                      | ✅ Easy (works with LangGraph etc.) |
| **Multi-step reasoning**           | ⚠️ Limited / brittle                  | ✅ Strong and structured            |
| **Production readiness**           | ❌ Not recommended now                 | ✅ Recommended standard             |
| **Prompt structure**               | Hidden defaults                       | Explicit prompt control            |
| **Tool argument passing**          | From parsed text                      | Direct structured arguments        |
| **Debugging**                      | ❌ Hard                                | ✅ Easier (clear steps)             |
